In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1C3VrBZQmHeKvJSKk4SpDsoL6pAkZLtpo", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/04_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Vector-Backed Memory: Embeddings, FAISS, and Semantic Retrieval

*Part 4 of the Vizuara series on Memory Architectures for LLM Applications*

**Estimated time: 60 minutes**

Every memory system we have built so far has a common limitation: retrieval is based on exact keywords. If the user stored "The budget is $5,000" and later asks "How much can I spend?", a keyword-based system will miss the connection entirely — the words "spend" and "budget" never overlap. Vector-backed memory solves this by converting text into numerical vectors (embeddings) that capture *meaning*. Similar meanings land near each other in vector space, so "budget" and "spend" end up close together even though they share no characters. In this notebook, we will build a complete vector memory system from scratch using sentence-transformers and FAISS, understand the mathematics of cosine similarity, and see how semantic retrieval outperforms keyword matching on realistic queries.

In [ ]:
#@title 🎧 Listen: Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_02_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you are searching through a library. The old approach is a card catalog — you look up a book by its exact title or author. If you search for "cooking Italian pasta", you will find "Italian Pasta Recipes" but miss "Traditional Noodle Dishes from Rome" even though it is about the same thing.

The new approach is semantic search. Instead of matching words, we match *meanings*. The library encodes every book as a point in a high-dimensional space, where nearby points represent similar topics. Now "cooking Italian pasta" and "Traditional Noodle Dishes from Rome" are neighbors — because they mean the same thing.

Vector-backed memory applies this idea to conversation history. Every message is embedded as a vector. When the user asks a question, we embed the query and find the most similar past messages using vector similarity. This means:
- "What is my budget?" retrieves "The budget is $5,000" (obvious keyword match)
- "How much can I spend?" *also* retrieves "The budget is $5,000" (semantic match!)
- "Tell me about the team composition" retrieves messages about Sarah, Alex, and Priya

This is the most powerful retrieval mechanism we have seen so far.

In this notebook, we will:
- Understand **embeddings** — how text becomes vectors
- Master **cosine similarity** — the math behind semantic matching
- Build a **FAISS-backed vector store** from scratch
- Implement **top-k retrieval** with similarity scoring
- Compare **keyword vs semantic retrieval** on the same queries
- Build a **complete vector memory system** for conversations

Let us begin.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

### What Are Embeddings?

An embedding is a function that maps text to a fixed-size numerical vector:

$$\text{embed}: \text{string} \rightarrow \mathbb{R}^d$$

where $d$ is the embedding dimension (typically 384, 768, or 1536). The magic is that this function is trained on billions of text examples, so it learns that semantically similar texts should have similar vectors.

Think of it this way: if we could somehow represent every possible meaning as a point in space, then "budget" and "spending money" would be close together, while "budget" and "kangaroo" would be far apart.

### How FAISS Works

FAISS (Facebook AI Similarity Search) is a library that efficiently finds the nearest neighbors in high-dimensional vector spaces. Without FAISS, finding the most similar vector among 1 million stored vectors would require computing 1 million similarity scores — O(n). FAISS uses indexing structures (like IVF and HNSW) to reduce this to O(log n) or better.

For our purposes, we will use `IndexFlatIP` (exact inner product search), which is simple and correct. For larger deployments, you would switch to approximate indexes that trade a tiny amount of accuracy for massive speed improvements.

### Cosine Similarity — The Core Metric

The key to vector retrieval is measuring how "similar" two vectors are. We use **cosine similarity**, which measures the cosine of the angle between two vectors:

- Cosine = 1: vectors point in exactly the same direction (identical meaning)
- Cosine = 0: vectors are orthogonal (unrelated meanings)
- Cosine = -1: vectors point in opposite directions (opposite meanings)

When vectors are normalized (length = 1), cosine similarity equals the dot product. This is why we normalize embeddings before storing them in FAISS — it lets us use the fast inner product search.

In [ ]:
#@title 🎧 Listen: The Mathematics
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_04_the_mathematics.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

### Cosine Similarity

Given two vectors $\mathbf{q}$ (query) and $\mathbf{m}$ (memory), cosine similarity is:

$$\text{sim}(\mathbf{q}, \mathbf{m}) = \frac{\mathbf{q} \cdot \mathbf{m}}{||\mathbf{q}|| \; ||\mathbf{m}||} = \frac{\sum_{i=1}^{d} q_i m_i}{\sqrt{\sum_{i=1}^{d} q_i^2} \cdot \sqrt{\sum_{i=1}^{d} m_i^2}}$$

**Worked example with 3D vectors:** Let $\mathbf{q} = [0.8, 0.6, 0.0]$ and $\mathbf{m} = [0.7, 0.7, 0.1]$:

$$\mathbf{q} \cdot \mathbf{m} = (0.8)(0.7) + (0.6)(0.7) + (0.0)(0.1) = 0.56 + 0.42 + 0 = 0.98$$

$$||\mathbf{q}|| = \sqrt{0.64 + 0.36 + 0} = 1.0$$

$$||\mathbf{m}|| = \sqrt{0.49 + 0.49 + 0.01} = \sqrt{0.99} = 0.995$$

$$\text{sim} = \frac{0.98}{1.0 \times 0.995} = 0.985$$

A similarity of 0.985 means these vectors are very similar — nearly identical in direction. This is exactly what we want when a query semantically matches a memory.

### Top-k Retrieval

Given a query vector $\mathbf{q}$ and $N$ stored memory vectors $\{\mathbf{m}_1, \ldots, \mathbf{m}_N\}$, we retrieve the top-$k$ most similar:

$$\mathcal{M}_{\text{retrieved}} = \underset{S \subseteq \{1,...,N\}, |S|=k}{\arg\max} \sum_{i \in S} \text{sim}(\mathbf{q}, \mathbf{m}_i)$$

In practice, FAISS computes all $N$ similarity scores and returns the indices of the $k$ highest. With normalized vectors, this is equivalent to finding the $k$ nearest neighbors by Euclidean distance.

**Worked example:** With $N = 5$ memories and scores $[0.85, 0.32, 0.91, 0.67, 0.45]$, top-3 retrieval returns indices $[2, 0, 3]$ (scores 0.91, 0.85, 0.67). These three memories are injected into the prompt.

### Token Budget for Vector Memory

The beauty of vector memory is that the token cost is constant regardless of conversation length:

$$C_{\text{vector}} = k \cdot \bar{t}_{\text{memory}} + t_{\text{recent}}$$

where $k$ is the number of retrieved memories and $\bar{t}_{\text{memory}}$ is the average tokens per memory.

**Worked example:** With $k = 5$ retrieved memories averaging 80 tokens each, and 1,000 tokens of recent messages:

$$C = 5 \times 80 + 1{,}000 = 1{,}400 \text{ tokens}$$

Whether the conversation has 50 messages or 50,000, we always retrieve exactly $k$ of them. The vector index handles millions of messages with sub-millisecond search time.

In [ ]:
#@title 🎧 Listen: Environment Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_05_environment_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Setting Up Our Environment

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu numpy matplotlib tiktoken

In [ ]:
import numpy as np
import faiss
import matplotlib.pyplot as plt
import tiktoken
import time
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer

ENCODER = tiktoken.encoding_for_model("gpt-4")

def count_tokens(text: str) -> int:
    return len(ENCODER.encode(text))

# Load the embedding model
print("Loading embedding model (all-MiniLM-L6-v2)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Model loaded! Embedding dimension: {EMBED_DIM}")

In [ ]:
#@title 🎧 Listen: Understanding Embeddings
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_06_understanding_embeddings.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let's Build It — Component by Component

### 4.1 Understanding Embeddings

Before building the memory system, let us develop intuition for what embeddings look like and how similarity works in practice.

In [ ]:
# Embed some sample sentences
sentences = [
    "The budget for this project is $5,000.",
    "How much money can I spend?",
    "What is the total cost?",
    "The weather in San Francisco is foggy.",
    "My favorite programming language is Python.",
    "I prefer writing code in Python.",
    "The deadline is March 15.",
    "When is the project due?",
]

print("Embedding sample sentences...")
embeddings = embed_model.encode(sentences)
print(f"Shape: {embeddings.shape}")
print(f"Each sentence -> {EMBED_DIM}-dimensional vector")

# Show first few dimensions of first embedding
print(f"\nFirst 10 dimensions of '{sentences[0][:40]}...':")
print(f"  {embeddings[0][:10]}")
print(f"  Vector norm: {np.linalg.norm(embeddings[0]):.4f}")

In [ ]:
#@title 🎧 Listen: Similarity Matrix Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_07_similarity_matrix_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Compute pairwise cosine similarity
from numpy.linalg import norm

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (norm(a) * norm(b))

# Build similarity matrix
n = len(sentences)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(
            embeddings[i], embeddings[j]
        )

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Cosine Similarity")

# Labels
short_labels = [s[:30] + "..." for s in sentences]
ax.set_xticks(range(n))
ax.set_xticklabels(short_labels, rotation=45, ha="right",
                    fontsize=8)
ax.set_yticks(range(n))
ax.set_yticklabels(short_labels, fontsize=8)
ax.set_title("Cosine Similarity Matrix\n"
             "Semantically similar sentences are green",
             fontsize=14, fontweight="bold")

# Annotate cells
for i in range(n):
    for j in range(n):
        color = "white" if sim_matrix[i, j] > 0.7 else "black"
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}",
                ha="center", va="center", fontsize=7,
                color=color)

plt.tight_layout()
plt.savefig("similarity_matrix.png", dpi=150,
            bbox_inches="tight")
plt.show()

print("\nKey observations:")
print(f"  'budget $5,000' vs 'how much money': "
      f"{sim_matrix[0, 1]:.3f} (semantic match!)")
print(f"  'budget $5,000' vs 'weather foggy': "
      f"{sim_matrix[0, 3]:.3f} (unrelated)")
print(f"  'Python language' vs 'code in Python': "
      f"{sim_matrix[4, 5]:.3f} (paraphrase)")
print(f"  'deadline March' vs 'project due': "
      f"{sim_matrix[6, 7]:.3f} (semantic match)")

In [ ]:
#@title 🎧 Listen: Post Viz Reinforce
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_08_post_viz_reinforce.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

This is exactly what we want. The embedding model captures that "How much money can I spend?" is semantically close to "The budget is $5,000" (high similarity), while "The weather in San Francisco is foggy" is unrelated (low similarity). Keyword matching would miss the first pair entirely because they share no keywords.

### 4.2 Building the FAISS Vector Store

Now let us build the vector store. FAISS handles the heavy lifting of nearest-neighbor search.

In [ ]:
#@title 🎧 Listen: Faiss Vector Store
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_09_faiss_vector_store.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class VectorStore:
    """FAISS-backed vector store for semantic retrieval.

    Stores text messages as normalized embeddings in a
    FAISS inner-product index. Supports top-k retrieval
    by cosine similarity.
    """

    def __init__(self, dim: int = EMBED_DIM):
        self.dim = dim
        self.index = faiss.IndexFlatIP(dim)  # Inner product
        self.messages: List[Dict] = []
        self.embeddings: List[np.ndarray] = []

    def add(self, text: str, metadata: Dict = None):
        """Add a text to the vector store."""
        # Compute and normalize embedding
        emb = embed_model.encode([text]).astype("float32")
        faiss.normalize_L2(emb)

        # Add to FAISS index
        self.index.add(emb)

        # Store message and embedding
        self.messages.append({
            "content": text,
            "tokens": count_tokens(text),
            "metadata": metadata or {},
        })
        self.embeddings.append(emb[0])

    def search(self, query: str, top_k: int = 5) -> List[Tuple[Dict, float]]:
        """Retrieve the top-k most similar messages.

        Returns list of (message_dict, similarity_score) tuples.
        """
        if self.index.ntotal == 0:
            return []

        # Embed and normalize query
        q_emb = embed_model.encode([query]).astype("float32")
        faiss.normalize_L2(q_emb)

        # Search
        k = min(top_k, self.index.ntotal)
        scores, indices = self.index.search(q_emb, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0 and idx < len(self.messages):
                results.append((self.messages[idx], float(score)))

        return results

    @property
    def size(self):
        return len(self.messages)

print("VectorStore defined!")

In [ ]:
#@title 🎧 Listen: Test Vector Store
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_10_test_vector_store.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Let us test it with a conversation:

In [ ]:
store = VectorStore()

# Add conversation messages
messages = [
    ("user", "My name is Raj and I am building an e-commerce chatbot."),
    ("assistant", "Hello Raj! What is your budget and timeline?"),
    ("user", "The budget is $5,000 and the deadline is March 15."),
    ("assistant", "Got it. $5,000 with a March 15 deadline."),
    ("user", "Sarah handles frontend with React. Alex does backend. "
     "Priya works on ML models."),
    ("assistant", "Great team. Sarah/React, Alex/backend, Priya/ML."),
    ("user", "We will use Python with FastAPI and PostgreSQL."),
    ("assistant", "Solid stack. FastAPI is fast and well-documented."),
    ("user", "We need OAuth2 authentication and Docker deployment."),
    ("assistant", "OAuth2 with JWT for auth. Docker on Cloud Run."),
    ("user", "I prefer dark mode and concise answers."),
    ("assistant", "Noted! I will keep responses concise."),
    ("user", "The chatbot needs to handle returns and order tracking."),
    ("assistant", "Returns processing and order tracking via API."),
]

for role, content in messages:
    store.add(content, metadata={"role": role})

print(f"Vector store populated: {store.size} messages")
print(f"FAISS index size: {store.index.ntotal} vectors")

# Test semantic retrieval
test_queries = [
    "How much can I spend?",           # Semantic: budget
    "Who is working on this project?",  # Semantic: team
    "What technology stack?",           # Semantic: tech
    "When must this be finished?",      # Semantic: deadline
    "What features does the chatbot need?", # Semantic: features
]

print(f"\n--- Semantic Retrieval Test ---")
for query in test_queries:
    results = store.search(query, top_k=2)
    print(f"\nQuery: \"{query}\"")
    for msg, score in results:
        preview = msg["content"][:60]
        print(f"  [{score:.3f}] {preview}...")

In [ ]:
#@title 🎧 Listen: Post Vector Store Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_11_post_vector_store_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Look at the results. "How much can I spend?" retrieves the message about the $5,000 budget even though the words are completely different. "When must this be finished?" finds the March 15 deadline. This is the power of semantic retrieval — it matches *meaning*, not keywords.

### 4.3 The Complete Vector Memory System

Now let us wrap the vector store into a full memory system with recent messages and a query interface.

In [ ]:
#@title 🎧 Listen: Complete Vector Memory System
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_12_complete_vector_memory_system.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class VectorMemory:
    """Complete vector-backed memory system.

    Combines semantic vector retrieval of older messages
    with a sliding window of recent messages (verbatim).
    """

    def __init__(
        self,
        recent_turns: int = 5,
        top_k: int = 5,
        system_prompt: str = "",
    ):
        self.store = VectorStore()
        self.recent_turns = recent_turns
        self.top_k = top_k
        self.system_prompt = system_prompt
        self.system_tokens = count_tokens(system_prompt)
        self.all_messages: List[Dict] = []
        self.turn_count = 0

        # Tracking
        self.context_history: List[int] = []
        self.retrieval_log: List[Dict] = []

    def add_message(self, role: str, content: str):
        """Add a message to both the vector store and
        the message history."""
        tokens = count_tokens(content)
        self.all_messages.append({
            "role": role,
            "content": content,
            "tokens": tokens,
        })

        # Add to vector store
        self.store.add(content, metadata={
            "role": role,
            "turn": self.turn_count,
        })

        if role == "user":
            self.turn_count += 1

        # Track context
        ctx_tokens = self._estimate_context_tokens()
        self.context_history.append(ctx_tokens)

    def retrieve(self, query: str) -> Dict:
        """Retrieve relevant context for a query.

        Returns:
            Dict with 'retrieved' (vector results),
            'recent' (last K turns), 'context' (combined text),
            and 'tokens' (total context tokens)
        """
        # Semantic retrieval from vector store
        retrieved = self.store.search(query, top_k=self.top_k)

        # Recent messages (sliding window)
        recent_count = self.recent_turns * 2
        recent = self.all_messages[-recent_count:]

        # Deduplicate: remove retrieved messages that are
        # already in the recent window
        recent_contents = set(m["content"] for m in recent)
        unique_retrieved = [
            (msg, score) for msg, score in retrieved
            if msg["content"] not in recent_contents
        ]

        # Build context string
        parts = []
        if self.system_prompt:
            parts.append(f"System: {self.system_prompt}")

        if unique_retrieved:
            ret_text = "\n".join(
                f"[Relevance {score:.2f}] {msg['content']}"
                for msg, score in unique_retrieved
            )
            parts.append(
                f"Relevant past messages:\n{ret_text}"
            )

        if recent:
            rec_text = "\n".join(
                f"{m['role']}: {m['content']}" for m in recent
            )
            parts.append(f"Recent conversation:\n{rec_text}")

        context = "\n\n".join(parts)
        total_tokens = count_tokens(context)

        self.retrieval_log.append({
            "query": query,
            "n_retrieved": len(unique_retrieved),
            "n_recent": len(recent),
            "total_tokens": total_tokens,
        })

        return {
            "retrieved": unique_retrieved,
            "recent": recent,
            "context": context,
            "tokens": total_tokens,
        }

    def _estimate_context_tokens(self) -> int:
        recent_count = self.recent_turns * 2
        recent = self.all_messages[-recent_count:]
        recent_tokens = sum(m["tokens"] for m in recent)
        # Estimate retrieved tokens as top_k * avg msg tokens
        avg_tokens = (
            sum(m["tokens"] for m in self.all_messages)
            / max(len(self.all_messages), 1)
        )
        retrieved_tokens = int(self.top_k * avg_tokens)
        return self.system_tokens + retrieved_tokens + recent_tokens

    def get_stats(self) -> Dict:
        return {
            "total_messages": len(self.all_messages),
            "vector_store_size": self.store.size,
            "total_turns": self.turn_count,
            "total_tokens": sum(
                m["tokens"] for m in self.all_messages
            ),
        }

print("VectorMemory system defined!")

In [ ]:
#@title 🎧 Listen: Test Vector Memory
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_13_test_vector_memory.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Let us populate it and test retrieval:

In [ ]:
vm = VectorMemory(
    recent_turns=3,
    top_k=3,
    system_prompt="You are a helpful project assistant.",
)

for role, content in messages:
    vm.add_message(role, content)

print(f"VectorMemory: {vm.get_stats()}")

# Test with a callback query
query = "How much money do I have for this project?"
result = vm.retrieve(query)

print(f"\nQuery: \"{query}\"")
print(f"\nRetrieved from vector store ({len(result['retrieved'])} items):")
for msg, score in result["retrieved"]:
    print(f"  [{score:.3f}] {msg['content'][:70]}...")

print(f"\nRecent messages ({len(result['recent'])} items):")
for m in result["recent"]:
    print(f"  [{m['role']:>9}] {m['content'][:60]}...")

print(f"\nTotal context tokens: {result['tokens']}")

In [ ]:
#@title 🎧 Listen: Todo 1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_14_todo_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn

### TODO 1: Implement Keyword vs Semantic Retrieval Comparison

Build a function that runs the same queries through both keyword-based and semantic retrieval, and computes the hit rate for each.

In [ ]:
#@title 🎧 Listen: Todo 1 Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_15_todo_1_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def compare_retrieval_methods(
    messages: List[Dict],
    test_queries: List[Dict],
    top_k: int = 3,
) -> Dict:
    """Compare keyword-based vs semantic retrieval.

    Args:
        messages: List of message dicts with 'role' and
            'content' keys
        test_queries: List of dicts with:
            - 'query': The search query
            - 'expected_keywords': List of keywords that
              should appear in at least one retrieved message
            - 'label': Short description of the test
        top_k: Number of results to retrieve

    Returns:
        Dict with keys:
            'keyword_results': list of bools (hit/miss per query)
            'semantic_results': list of bools (hit/miss per query)
            'keyword_accuracy': fraction of hits
            'semantic_accuracy': fraction of hits
            'per_query': list of dicts with details per query

    Keyword retrieval algorithm:
        1. Tokenize query into lowercase words
        2. Score each message by |query_words & msg_words|
        3. Return top-k highest-scoring messages

    Semantic retrieval:
        Use the VectorStore.search() method

    Hit condition:
        At least one retrieved message contains ALL of the
        expected_keywords (case-insensitive)

    Hints:
        - Build a VectorStore and populate it with messages
        - For keyword scoring, use set intersection
        - Check if expected_keywords all appear in
          msg['content'].lower()
    """
    # ---- YOUR CODE HERE ---- #

    results = {
        "keyword_results": [],
        "semantic_results": [],
        "keyword_accuracy": 0.0,
        "semantic_accuracy": 0.0,
        "per_query": [],
    }

    # TODO: Build vector store from messages
    # TODO: For each query, run both retrieval methods
    # TODO: Check hit condition
    # TODO: Compute accuracies

    return results

    # ---- END YOUR CODE ---- #

In [ ]:
#@title 🎧 Listen: Todo 2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_16_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Implement a Similarity Threshold Filter

In our current system, we always retrieve exactly top-k results regardless of how similar they actually are. If the query is "Tell me a joke" and the conversation was entirely about project planning, the top-3 results will still be returned even though they are irrelevant. Implement a threshold-based filter.

In [ ]:
#@title 🎧 Listen: Todo 2 Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_17_todo_2_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class ThresholdVectorStore(VectorStore):
    """Vector store with a minimum similarity threshold.

    Only returns results above a configurable similarity
    threshold, preventing irrelevant retrievals.

    Args:
        dim: Embedding dimension
        min_similarity: Minimum cosine similarity for a
            result to be included (default: 0.3)
    """

    def __init__(self, dim: int = EMBED_DIM,
                 min_similarity: float = 0.3):
        super().__init__(dim)
        self.min_similarity = min_similarity

    def search_with_threshold(
        self, query: str, top_k: int = 5
    ) -> List[Tuple[Dict, float]]:
        """Retrieve top-k results above the similarity threshold.

        Requirements:
            1. Call the parent search() with top_k
            2. Filter results to only include those with
               similarity >= self.min_similarity
            3. Return the filtered list

        Additionally, track:
            - Total candidates (before filtering)
            - Retained candidates (after filtering)
            - Max similarity score
            - Min similarity score (among returned)

        Returns:
            List of (message, score) tuples, filtered by
            threshold. May be shorter than top_k if some
            results fall below the threshold.

        Example:
            >>> store = ThresholdVectorStore(min_similarity=0.5)
            >>> store.add("The budget is $5,000")
            >>> store.add("The weather is nice today")
            >>> results = store.search_with_threshold(
            ...     "How much money?", top_k=5
            ... )
            >>> # Only budget message returned (weather < 0.5)
        """
        # ---- YOUR CODE HERE ---- #

        # TODO: Call parent search
        # TODO: Filter by threshold
        # TODO: Return filtered results

        return []

        # ---- END YOUR CODE ---- #

In [ ]:
#@title 🎧 Listen: After Todo 1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_18_after_todo_1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Verification Cell for TODO 1

In [ ]:
#@title 🎧 Listen: Viz Keyword Vs Semantic
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_25_viz_keyword_vs_semantic.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Test compare_retrieval_methods ---
test_msgs = [
    {"role": "user", "content": "My name is Raj and I work at Google."},
    {"role": "assistant", "content": "Hello Raj! How can I help?"},
    {"role": "user", "content": "The budget is $5,000 for the project."},
    {"role": "assistant", "content": "Got it. $5,000 budget noted."},
    {"role": "user", "content": "We are using Python and FastAPI."},
    {"role": "assistant", "content": "Python with FastAPI is great."},
    {"role": "user", "content": "The deadline is March 15."},
    {"role": "assistant", "content": "March 15 deadline confirmed."},
    {"role": "user", "content": "I prefer dark mode interfaces."},
    {"role": "assistant", "content": "Dark mode preference noted."},
]

test_qs = [
    {"query": "How much money can I spend?",
     "expected_keywords": ["5,000"],
     "label": "budget (semantic)"},
    {"query": "What is the budget?",
     "expected_keywords": ["5,000"],
     "label": "budget (keyword)"},
    {"query": "When must this be finished?",
     "expected_keywords": ["march"],
     "label": "deadline (semantic)"},
    {"query": "What technology stack are we using?",
     "expected_keywords": ["python"],
     "label": "tech (semantic)"},
    {"query": "What programming language?",
     "expected_keywords": ["python"],
     "label": "tech (keyword)"},
]

comp = compare_retrieval_methods(test_msgs, test_qs, top_k=3)

assert "keyword_accuracy" in comp
assert "semantic_accuracy" in comp
assert len(comp["keyword_results"]) == len(test_qs)
assert len(comp["semantic_results"]) == len(test_qs)

print(f"Keyword accuracy: {comp['keyword_accuracy']:.0%}")
print(f"Semantic accuracy: {comp['semantic_accuracy']:.0%}")

for detail in comp["per_query"]:
    kw = "HIT" if detail["keyword_hit"] else "MISS"
    sem = "HIT" if detail["semantic_hit"] else "MISS"
    print(f"  [{kw:>4} / {sem:>4}] {detail['label']}")

assert comp["semantic_accuracy"] >= comp["keyword_accuracy"], \
    "Semantic should be >= keyword accuracy"

print("\nAll retrieval comparison tests passed!")

In [ ]:
#@title 🎧 Listen: After Todo 2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_19_after_todo_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Verification Cell for TODO 2

In [ ]:
# --- Test ThresholdVectorStore ---
ts = ThresholdVectorStore(min_similarity=0.4)

ts.add("The project budget is $5,000.")
ts.add("Sarah works on the frontend with React.")
ts.add("The weather in Hawaii is beautiful.")
ts.add("Python is a popular programming language.")

# Query that matches budget
results_budget = ts.search_with_threshold(
    "How much money do we have?", top_k=5
)
print(f"Query about money: {len(results_budget)} results")
for msg, score in results_budget:
    print(f"  [{score:.3f}] {msg['content'][:50]}...")

# Query that should match few/no results
results_random = ts.search_with_threshold(
    "Tell me a joke about elephants", top_k=5
)
print(f"\nQuery about elephants: {len(results_random)} results")

# Budget query should return results
assert len(results_budget) >= 1, "Budget query should find matches"

# All results should be above threshold
for msg, score in results_budget:
    assert score >= ts.min_similarity, \
        f"Score {score} below threshold {ts.min_similarity}"

for msg, score in results_random:
    assert score >= ts.min_similarity, \
        f"Score {score} below threshold {ts.min_similarity}"

print("\nAll ThresholdVectorStore tests passed!")

In [ ]:
#@title 🎧 Listen: Solution Reveal
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_20_solution_reveal.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

<details>
<summary><b>Solution: compare_retrieval_methods</b></summary>

In [ ]:
def compare_retrieval_methods(messages, test_queries, top_k=3):
    # Build vector store
    vs = VectorStore()
    for msg in messages:
        vs.add(msg["content"], metadata={"role": msg["role"]})

    keyword_results = []
    semantic_results = []
    per_query = []

    for tq in test_queries:
        query = tq["query"]
        expected = tq["expected_keywords"]

        # Keyword retrieval
        query_words = set(query.lower().split())
        scored = []
        for msg in messages:
            msg_words = set(msg["content"].lower().split())
            score = len(query_words & msg_words)
            scored.append((msg, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        kw_top = [msg for msg, _ in scored[:top_k]]

        kw_hit = any(
            all(kw.lower() in msg["content"].lower() for kw in expected)
            for msg in kw_top
        )
        keyword_results.append(kw_hit)

        # Semantic retrieval
        sem_top = vs.search(query, top_k=top_k)
        sem_hit = any(
            all(kw.lower() in msg["content"].lower() for kw in expected)
            for msg, _ in sem_top
        )
        semantic_results.append(sem_hit)

        per_query.append({
            "label": tq["label"],
            "keyword_hit": kw_hit,
            "semantic_hit": sem_hit,
        })

    n = len(test_queries)
    return {
        "keyword_results": keyword_results,
        "semantic_results": semantic_results,
        "keyword_accuracy": sum(keyword_results) / n,
        "semantic_accuracy": sum(semantic_results) / n,
        "per_query": per_query,
    }

</details>

<details>
<summary><b>Solution: ThresholdVectorStore.search_with_threshold</b></summary>

In [ ]:
def search_with_threshold(self, query, top_k=5):
    all_results = super().search(query, top_k=top_k)
    filtered = [
        (msg, score) for msg, score in all_results
        if score >= self.min_similarity
    ]
    return filtered

</details>

In [ ]:
#@title 🎧 Listen: Load Solutions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_21_load_solutions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Load solutions ---
def compare_retrieval_methods(messages, test_queries, top_k=3):
    vs = VectorStore()
    for msg in messages:
        vs.add(msg["content"], metadata={"role": msg["role"]})
    keyword_results = []
    semantic_results = []
    per_query = []
    for tq in test_queries:
        query = tq["query"]
        expected = tq["expected_keywords"]
        query_words = set(query.lower().split())
        scored = []
        for msg in messages:
            msg_words = set(msg["content"].lower().split())
            score = len(query_words & msg_words)
            scored.append((msg, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        kw_top = [msg for msg, _ in scored[:top_k]]
        kw_hit = any(
            all(kw.lower() in msg["content"].lower() for kw in expected)
            for msg in kw_top
        )
        keyword_results.append(kw_hit)
        sem_top = vs.search(query, top_k=top_k)
        sem_hit = any(
            all(kw.lower() in msg["content"].lower() for kw in expected)
            for msg, _ in sem_top
        )
        semantic_results.append(sem_hit)
        per_query.append({
            "label": tq["label"], "keyword_hit": kw_hit, "semantic_hit": sem_hit,
        })
    n = len(test_queries)
    return {
        "keyword_results": keyword_results,
        "semantic_results": semantic_results,
        "keyword_accuracy": sum(keyword_results) / n,
        "semantic_accuracy": sum(semantic_results) / n,
        "per_query": per_query,
    }

class ThresholdVectorStore(VectorStore):
    def __init__(self, dim=EMBED_DIM, min_similarity=0.3):
        super().__init__(dim)
        self.min_similarity = min_similarity

    def search_with_threshold(self, query, top_k=5):
        all_results = super().search(query, top_k=top_k)
        return [(msg, score) for msg, score in all_results
                if score >= self.min_similarity]

print("Solutions loaded!")

In [ ]:
#@title 🎧 Listen: Putting It All Together
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_22_putting_it_all_together.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together

Let us build and run the complete vector memory system on a long conversation with topic callbacks.

In [ ]:
#@title 🎧 Listen: Full System Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_23_full_system_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Full conversation with topic phases
full_vm = VectorMemory(
    recent_turns=3,
    top_k=3,
    system_prompt="You are a project assistant.",
)

conversation = [
    # Phase 1: Project setup (turns 1-3)
    ("user", "My name is Raj. Budget is $5,000, deadline March 15."),
    ("assistant", "Hello Raj! $5,000 budget and March 15 deadline noted."),
    ("user", "Team: Sarah (frontend/React), Alex (backend/FastAPI), Priya (ML)."),
    ("assistant", "Great team of 4 with clear role assignments."),
    ("user", "We need PostgreSQL with pgvector for vector search."),
    ("assistant", "PostgreSQL with pgvector is excellent for this."),
    # Phase 2: Implementation details (turns 4-6)
    ("user", "The chatbot needs to handle returns and order tracking."),
    ("assistant", "Two core features: returns and order tracking."),
    ("user", "We need OAuth2 for authentication with JWT tokens."),
    ("assistant", "OAuth2 with JWT is the standard approach."),
    ("user", "Deploy with Docker on Google Cloud Run."),
    ("assistant", "Docker + Cloud Run for easy scaling."),
    # Phase 3: Testing and monitoring (turns 7-9)
    ("user", "Use pytest for testing with 80% coverage target."),
    ("assistant", "pytest with 80% coverage is a good target."),
    ("user", "Prometheus and Grafana for monitoring."),
    ("assistant", "Prometheus + Grafana for metrics and dashboards."),
    ("user", "Set up error rate alerts above 1%."),
    ("assistant", "1% error rate threshold for alerting."),
    # Phase 4: Callbacks (turn 10)
    ("user", "Going back to the beginning — how much can I spend "
     "and who is on my team?"),
]

for role, content in conversation:
    full_vm.add_message(role, content)

# Retrieve for the callback query
query = "How much can I spend and who is on my team?"
result = full_vm.retrieve(query)

print("=" * 60)
print("VECTOR MEMORY: SEMANTIC CALLBACK RETRIEVAL")
print("=" * 60)

print(f"\nQuery: \"{query}\"")
print(f"\nRetrieved memories (semantic search):")
for msg, score in result["retrieved"]:
    print(f"  [{score:.3f}] {msg['content'][:70]}...")

print(f"\nRecent messages:")
for m in result["recent"]:
    print(f"  [{m['role']:>9}] {m['content'][:60]}...")

print(f"\nTotal context: {result['tokens']} tokens")
print(f"Total conversation: {full_vm.get_stats()['total_tokens']} tokens")

In [ ]:
#@title 🎧 Listen: Training And Results Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_24_training_and_results_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Training and Results

### Visualization 1: Semantic Retrieval vs Keyword Matching

In [ ]:
comp = compare_retrieval_methods(
    [{"role": r, "content": c} for r, c in conversation],
    [
        {"query": "How much money can I spend?",
         "expected_keywords": ["5,000"], "label": "Budget (semantic)"},
        {"query": "What is the budget?",
         "expected_keywords": ["5,000"], "label": "Budget (keyword)"},
        {"query": "When must the project be finished?",
         "expected_keywords": ["march"], "label": "Deadline (semantic)"},
        {"query": "Who is doing the frontend work?",
         "expected_keywords": ["sarah"], "label": "Team (semantic)"},
        {"query": "What tech stack are we using?",
         "expected_keywords": ["python", "fastapi"],
         "label": "Stack (semantic)"},
        {"query": "How do we track things going wrong?",
         "expected_keywords": ["monitor", "prometheus"],
         "label": "Monitoring (semantic)"},
        {"query": "What is the testing strategy?",
         "expected_keywords": ["pytest"], "label": "Testing (semantic)"},
        {"query": "Where are we hosting this?",
         "expected_keywords": ["cloud", "docker"],
         "label": "Deploy (semantic)"},
    ],
    top_k=3,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Per-query comparison
ax1 = axes[0]
labels = [d["label"] for d in comp["per_query"]]
kw_hits = [1 if d["keyword_hit"] else 0 for d in comp["per_query"]]
sem_hits = [1 if d["semantic_hit"] else 0 for d in comp["per_query"]]

x = np.arange(len(labels))
width = 0.35
ax1.barh(x - width/2, kw_hits, width, label="Keyword",
         color="#F59E0B", alpha=0.8)
ax1.barh(x + width/2, sem_hits, width, label="Semantic",
         color="#3B82F6", alpha=0.8)
ax1.set_yticks(x)
ax1.set_yticklabels(labels, fontsize=9)
ax1.set_xlabel("Hit (1) or Miss (0)", fontsize=12)
ax1.set_title("Per-Query Retrieval Results",
              fontsize=14, fontweight="bold")
ax1.legend(fontsize=10)
ax1.set_xlim(0, 1.3)

# Right: Overall accuracy
ax2 = axes[1]
methods = ["Keyword\nMatching", "Semantic\n(Embeddings)"]
accuracies = [comp["keyword_accuracy"], comp["semantic_accuracy"]]
colors = ["#F59E0B", "#3B82F6"]
bars = ax2.bar(methods, accuracies, color=colors, width=0.5,
               edgecolor="white")
for bar, acc in zip(bars, accuracies):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.02,
             f"{acc:.0%}", ha="center", fontsize=16,
             fontweight="bold")
ax2.set_ylim(0, 1.15)
ax2.set_ylabel("Retrieval Accuracy", fontsize=12)
ax2.set_title("Overall Accuracy Comparison",
              fontsize=14, fontweight="bold")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("keyword_vs_semantic.png", dpi=150,
            bbox_inches="tight")
plt.show()

print(f"\nSemantic retrieval: {comp['semantic_accuracy']:.0%}")
print(f"Keyword matching: {comp['keyword_accuracy']:.0%}")

In [ ]:
#@title 🎧 Listen: Viz Embedding Space
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_26_viz_embedding_space.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization 2: Embedding Space (2D Projection)

Let us visualize how conversation messages cluster in embedding space.

In [ ]:
from sklearn.decomposition import PCA

# Get all embeddings
all_embs = np.array(full_vm.store.embeddings)
all_texts = [m["content"][:40] for m in full_vm.store.messages]
all_roles = [m["metadata"]["role"] for m in full_vm.store.messages]

# Project to 2D
pca = PCA(n_components=2)
embs_2d = pca.fit_transform(all_embs)

# Also project the query
q_emb = embed_model.encode([query]).astype("float32")
faiss.normalize_L2(q_emb)
q_2d = pca.transform(q_emb)

fig, ax = plt.subplots(figsize=(14, 10))

# Color by role
role_colors = {"user": "#3B82F6", "assistant": "#10B981"}
for role, color in role_colors.items():
    mask = [r == role for r in all_roles]
    pts = embs_2d[mask]
    ax.scatter(pts[:, 0], pts[:, 1], c=color, s=100,
               alpha=0.7, label=f"{role.capitalize()} messages",
               edgecolors="white", linewidth=1)

# Highlight retrieved messages
retrieved_contents = set(
    msg["content"] for msg, _ in result["retrieved"]
)
for i, msg in enumerate(full_vm.store.messages):
    if msg["content"] in retrieved_contents:
        ax.scatter(embs_2d[i, 0], embs_2d[i, 1],
                   c="gold", s=250, marker="*",
                   edgecolors="black", linewidth=1.5,
                   zorder=5)

# Plot query
ax.scatter(q_2d[0, 0], q_2d[0, 1], c="red", s=200,
           marker="D", edgecolors="black", linewidth=2,
           zorder=5, label="Query")

# Add labels
for i, (pt, text) in enumerate(zip(embs_2d, all_texts)):
    ax.annotate(text[:25] + "...", xy=pt, fontsize=6,
                alpha=0.7, xytext=(5, 5),
                textcoords="offset points")

ax.set_xlabel("PCA Dimension 1", fontsize=12)
ax.set_ylabel("PCA Dimension 2", fontsize=12)
ax.set_title("Conversation Messages in Embedding Space\n"
             "Stars = Retrieved, Diamond = Query",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10, loc="upper right")
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("embedding_space.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
#@title 🎧 Listen: Viz Search Latency
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_27_viz_search_latency.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization 3: Search Latency vs Store Size

In [ ]:
# Benchmark FAISS search speed at different store sizes
sizes = [10, 50, 100, 500, 1000, 5000]
latencies = []

for size in sizes:
    test_store = VectorStore()
    # Add dummy messages
    for i in range(size):
        test_store.add(f"This is message number {i} about topic {i % 20}.")

    # Benchmark search
    q = "What is the budget for the project?"
    times = []
    for _ in range(10):
        start = time.time()
        test_store.search(q, top_k=5)
        elapsed = (time.time() - start) * 1000  # ms
        times.append(elapsed)

    avg_ms = np.mean(times)
    latencies.append(avg_ms)
    print(f"  Store size: {size:>5} | Search: {avg_ms:.2f} ms")

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(sizes, latencies, "o-", color="#3B82F6",
        linewidth=2.5, markersize=8)
ax.set_xlabel("Number of Messages in Store", fontsize=12)
ax.set_ylabel("Search Latency (ms)", fontsize=12)
ax.set_title("FAISS Search Latency vs Store Size",
             fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)

# Add annotation
ax.annotate(f"{latencies[-1]:.1f} ms at {sizes[-1]:,} msgs",
            xy=(sizes[-1], latencies[-1]),
            xytext=(sizes[-1] - 1500, latencies[-1] + 1),
            fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="->"))

plt.tight_layout()
plt.savefig("search_latency.png", dpi=150,
            bbox_inches="tight")
plt.show()

print("\nFAISS is remarkably fast — even with thousands of")
print("messages, search takes only a few milliseconds.")

In [ ]:
#@title 🎧 Listen: Final Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_28_final_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Final Output

Let us demonstrate the complete system handling a multi-topic conversation with semantic callbacks.

In [ ]:
print("=" * 60)
print("  FINAL DEMO: SEMANTIC MEMORY RETRIEVAL")
print("=" * 60)

demo_queries = [
    ("How much can I spend on this?",
     "Budget: the user mentioned $5,000"),
    ("Who is responsible for the ML work?",
     "Team: Priya handles ML models"),
    ("When do we need to deliver?",
     "Timeline: March 15 deadline"),
    ("What are we using for observability?",
     "Monitoring: Prometheus and Grafana"),
    ("What level of test coverage are we aiming for?",
     "Testing: 80% coverage with pytest"),
]

print()
for query, expected_context in demo_queries:
    result = full_vm.retrieve(query)
    top_msg, top_score = result["retrieved"][0] if result["retrieved"] else ({}, 0)

    print(f"Q: \"{query}\"")
    print(f"  Expected: {expected_context}")
    if top_msg:
        print(f"  Retrieved [{top_score:.3f}]: "
              f"{top_msg['content'][:65]}...")
    print()

# Summary statistics
stats = full_vm.get_stats()
print("=" * 60)
print(f"Store: {stats['vector_store_size']} vectors")
print(f"Total conversation: {stats['total_tokens']} tokens")
print(f"Context per query: ~{result['tokens']} tokens (bounded!)")
print(f"\nVector memory retrieves semantically relevant context")
print(f"regardless of exact keyword matches. Cost stays bounded")
print(f"at O(k) regardless of conversation length.")
print("=" * 60)

In [ ]:
#@title 🎧 Listen: Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_29_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### What Did We Build?

1. **Embedding pipeline** — converting text to 384-dimensional vectors using sentence-transformers
2. **FAISS vector store** — efficient nearest-neighbor search with normalized cosine similarity
3. **Semantic retrieval** — finding relevant memories by meaning, not keywords
4. **Threshold filtering** — preventing irrelevant results from polluting the context
5. **Complete vector memory** — combining semantic retrieval with a recent message window

### Key Insights

1. **Semantic > Keyword**: Embeddings capture meaning, so "How much can I spend?" matches "The budget is $5,000" even with zero keyword overlap. This is a fundamental improvement over all previous approaches.

2. **Constant cost**: Vector memory always retrieves exactly $k$ results, so the context size is bounded at $k \cdot \bar{t} + t_{\text{recent}}$ regardless of conversation length.

3. **FAISS is fast**: Even with thousands of vectors, search takes only milliseconds. This scales to production workloads.

4. **Threshold filtering matters**: Without a similarity threshold, every query returns $k$ results even when none are relevant. The threshold prevents irrelevant context from confusing the LLM.

### Questions to Think About

1. What happens when two messages have high similarity to the query but contain contradictory information? (e.g., "Budget is $5,000" and "Actually, budget is now $8,000") How would you handle this?

2. Embeddings treat all text equally, but some messages are more important than others. How would you weight importance? (Hint: think about combining similarity with a separate importance score.)

3. Our embeddings are static — computed once when the message is stored. But the meaning of a message can change based on later context. Should embeddings be updated over time?

4. Could you use multiple embedding models? For example, one optimized for factual retrieval and another for emotional/social context?

5. What is the failure mode of vector retrieval? When would keyword matching actually perform *better*?

### What Comes Next

We have now built four memory systems: buffer, sliding window, summary, entity, and vector. Each has strengths and weaknesses. In the final notebook, we will **combine them all** into a hybrid memory architecture — the most powerful approach that uses each system for what it does best.

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_30_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("Notebook complete!")
print()
print("Key concepts:")
print("  Embeddings: text -> high-dimensional vectors")
print("  Cosine similarity: sim(q, m) = q.m / (||q|| ||m||)")
print("  FAISS: efficient nearest-neighbor search")
print("  Top-k retrieval: bounded context, semantic matching")
print()
print("The fundamental insight: meaning > keywords.")
print("Next up: Hybrid Memory — combining all four approaches.")